In [1]:
import altair as alt
import matplotlib.pyplot as plt
import polars as pl

from traffic_balve.create_df import create_df

In [2]:
df = (
    create_df()
    .sort("datetime")
    .with_columns(
        delay_due_to_traffic_min=(
            pl.col("duration_in_traffic_s") - pl.col("duration_s")
        )
        / 60,
        delay_due_to_traffic_percent=100
        * (pl.col("duration_in_traffic_s") - pl.col("duration_s"))
        / pl.col("duration_s"),
    )
    .sort("datetime")
    .rolling(
        index_column="datetime",
        period="1m",
        by=["from_to"],
    )
    .agg(
        pl.col(
            "delay_due_to_traffic_percent",
            "delay_due_to_traffic_min",
            "duration_in_traffic_s",
            "duration_s",
        ).mean()
    )
    # .with_columns(
    #     date=pl.col("datetime").dt.date(),
    #     time=pl.col("datetime").dt.time(),
    # )
)
print(df.shape)
df.head(5)

(6078, 6)


from_to,datetime,delay_due_to_traffic_percent,delay_due_to_traffic_min,duration_in_traffic_s,duration_s
str,datetime[ns],f64,f64,f64,f64
"""Höhle -> Krank…",2023-12-20 10:50:43.063641,30.21978,0.916667,237.0,182.0
"""Höhle -> Krank…",2023-12-20 10:59:09.884939,25.824176,0.783333,229.0,182.0
"""Höhle -> Krank…",2023-12-20 11:09:56.003331,17.582418,0.533333,214.0,182.0
"""Höhle -> Krank…",2023-12-20 11:12:42.347660,17.582418,0.533333,214.0,182.0
"""Höhle -> Krank…",2023-12-20 11:17:40.084821,17.032967,0.516667,213.0,182.0


In [3]:
alt.data_transformers.disable_max_rows()


base = (
    alt.Chart(
        df  # type:ignore
    )
    .mark_line(strokeWidth=3, point=True)
    .encode(
        x=alt.X("hoursminutes(datetime):O").title("Uhrzeit"),
        detail=alt.Detail("monthdate(datetime):O"),
        color=alt.Color("from_to:N")
        .title("Richtung")
        .scale(
            domain=[
                "Höhle -> Krankenhaus",
                "Krankenhaus -> Höhle",
                "Höhle -> Krumpaul",
                "Krumpaul -> Höhle",
                "Krankenhaus -> Krumpaul",
                "Krumpaul -> Krankenhaus",
            ],
            range=[
                "#1F77B4",
                "#AEC7E8",
                "#FF7F0E",
                "#FFBB78",
                "#2CA02C",
                "#98DF8A",
            ],
        ),
    )
    .properties(width=1500, height=250)
)


# alt.layer(
#     base.encode(
#         y=alt.Y("duration_in_traffic_s:Q").title("Reisezeit [Sekunden]"),
#     ),
#     base.mark_line(strokeWidth=2, strokeDash=[2, 2]).encode(
#         y=alt.Y("duration_s:Q").title("Reisezeit [Sekunden]"),
#     ),
# )
alt.layer(
    base.encode(
        y=alt.Y("duration_in_traffic_s:Q").title("Reisezeit [Sekunden]"),
    ),
).facet(row="from_to:N")

alt.FacetChart(...)

In [4]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

tmp = (
    create_df()
    .sort("datetime")
    .select(
        "datetime",
        "from_to",
        "duration_in_traffic_s",
    )
    .pivot(index="datetime", columns="from_to", values="duration_in_traffic_s")
)

axs[0].scatter(
    tmp["Höhle -> Krankenhaus"],
    tmp["Krankenhaus -> Höhle"],
    color="blue",
)
axs[1].scatter(
    tmp["Höhle -> Krumpaul"],
    tmp["Krumpaul -> Höhle"],
    color="orange",
)
axs[2].scatter(
    tmp["Krankenhaus -> Krumpaul"],
    tmp["Krumpaul -> Krankenhaus"],
    color="green",
)

for ax in axs:
    ax.grid()
    ax.plot([100, 300], [100, 300], color="gray", zorder=-1)
axs[0].set(
    xlabel="Höhle -> Krankenhaus",
    ylabel="Krankenhaus -> Höhle",
    xlim=(100, 300),
    ylim=(100, 300),
)
axs[1].set(
    xlabel="Höhle -> Krumpaul",
    ylabel="Krumpaul -> Höhle",
    xlim=(100, 300),
    ylim=(100, 300),
)
axs[2].set(
    xlabel="Krankenhaus -> Krumpaul",
    ylabel="Krumpaul -> Krankenhaus",
    xlim=(100, 300),
    ylim=(100, 300),
)

[Text(0.5, 0, 'Krankenhaus -> Krumpaul'),
 Text(0, 0.5, 'Krumpaul -> Krankenhaus'),
 (100.0, 300.0),
 (100.0, 300.0)]

Error in callback <function flush_figures at 0x13182bb50> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [5]:
(
    create_df()
    .sort("datetime")
    .select(
        "datetime",
        "from_to",
        "duration_in_traffic_s",
    )
    .pivot(index="datetime", columns="from_to", values="duration_in_traffic_s")
)

datetime,Höhle -> Krankenhaus,Höhle -> Krumpaul,Krankenhaus -> Höhle,Krankenhaus -> Krumpaul,Krumpaul -> Höhle,Krumpaul -> Krankenhaus
datetime[ns],i64,i64,i64,i64,i64,i64
2023-12-20 10:50:43.063641,237,221,209,177,209,175
2023-12-20 10:59:09.884939,229,219,213,179,231,189
2023-12-20 11:09:56.003331,214,206,205,170,212,175
2023-12-20 11:12:42.347660,214,206,205,170,213,175
2023-12-20 11:17:40.084821,213,200,203,169,204,168
2023-12-20 11:31:14.957092,228,222,211,180,203,168
2023-12-20 11:46:08.886872,211,209,199,170,205,164
2023-12-20 12:00:02.024328,202,198,198,168,202,165
2023-12-20 12:15:01.910446,255,251,206,177,201,164
